In [1]:
import polars as pl
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoModel, 
    AutoTokenizer, 
    Trainer, 
    TrainingArguments,
    BertForSequenceClassification,
    BertTokenizer
)
from sklearn.preprocessing import LabelEncoder
from tqdm import tqdm
from huggingface_hub import snapshot_download
import warnings
import math


warnings.filterwarnings('ignore')

In [2]:
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

ACCEPTABLE_ACTIONS = ["view", "click", "clickout", "like"]
MICROS_IN_DAY = 24 * 60 * 60 * 1_000_000
TOP_K = 15
BATCH_SIZE_TRAIN = 5000      
MAX_SEQ_LEN = 30             
MIN_SEQ_LEN = 3
TOP_N_ITEMS = 5000          
SAMPLE_USERS = 5000         
SAMPLE_RATIO = 0.1 


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
snapshot_download(
    repo_id="t-tech/T-ECD",
    repo_type="dataset",
    local_dir=".",
    local_dir_use_symlinks=False,
    allow_patterns=["dataset/small/users.pq", "dataset/small/marketplace/**"]
)

Fetching ... files: 0it [00:00, ?it/s]

'/kaggle/working'

In [4]:
events_path = "/kaggle/working/dataset/small/marketplace/events"
items_path = "/kaggle/working/dataset/small/marketplace/items.pq"

events_lf = pl.scan_parquet(events_path).select([
    "timestamp", "user_id", "item_id", "action_type", "subdomain", "os"
])

In [5]:
items_lf = pl.scan_parquet(items_path).select([
    "item_id", "brand_id", "category", "subcategory", "price"
])

In [6]:
items_lf = items_lf.drop_nulls(subset=["price"]).with_columns([
    pl.col("category").fill_null("unknown"),
    pl.col("subcategory").fill_null("unknown"),
    pl.col("brand_id").fill_null(-1),
])


In [7]:
top_items = (
    events_lf
    .group_by("item_id")
    .agg(pl.len().alias("cnt"))
    .sort("cnt", descending=True)
    .limit(TOP_N_ITEMS)
    .select("item_id")
    .collect()
)["item_id"].to_list()

In [8]:
events_filtered = (
    events_lf
    .filter(pl.col("item_id").is_in(top_items))
    .filter(pl.col("action_type").is_in(["view", "click", "clickout", "like"]))
    .collect(streaming=True)  
)

In [9]:
action_counts = events_filtered.group_by("action_type").agg(pl.len())
total_events = action_counts["len"].sum()

action_weights = {}
for row in action_counts.iter_rows(named=True):
    weight = math.log(total_events / row["len"]) + 1
    action_weights[row["action_type"]] = weight

In [10]:
events_filtered = events_filtered.with_columns([
    pl.col("action_type").replace(action_weights).alias("target")
])

In [11]:
all_users = events_filtered["user_id"].unique().to_list()
n_users = len(all_users)

if len(all_users) > SAMPLE_USERS:
    np.random.seed(SEED)
    sampled_users = np.random.choice(all_users, SAMPLE_USERS, replace=False)
    events_filtered = events_filtered.filter(pl.col("user_id").is_in(sampled_users))

In [12]:
def global_temporal_split(df_lf: pl.LazyFrame, test_size: float = 0.2):
    """
    Временное разделение данных на train и test.
    """
    # Находим границы времени
    min_ts, max_ts = (
        df_lf
        .select(
            pl.col("timestamp").min().alias("min_ts"),
            pl.col("timestamp").max().alias("max_ts")
        )
        .collect()
        .row(0)
    )
    
    # Вычисляем пороговое значение для разделения
    ts_range = max_ts - min_ts
    split_ts = min_ts + ts_range * (1 - test_size)
    
    # Разделяем данные
    train_lf = df_lf.filter(pl.col("timestamp") < split_ts)
    test_lf = df_lf.filter(pl.col("timestamp") >= split_ts)
    
    return train_lf, test_lf

In [13]:
def ndcg_at_k_vectorized(
    user_based_df: pl.DataFrame,
    relevancy_col: str,
    true_items_col: str,
    predicted_items_col: str,
    predicted_score_col: str,
    top_k: int = 15,
) -> float:
    """
    Вычисление NDCG@k с использованием векторизованных операций polars.
    """
    _ITEM = "__item"
    _RANK = "__rank"

    truth = (
        user_based_df.select("user_id", true_items_col, relevancy_col)
        .explode(true_items_col, relevancy_col)
        .group_by("user_id", true_items_col)
        .agg(pl.col(relevancy_col).max())
        .rename({true_items_col: _ITEM})
    )

    preds = (
        user_based_df.select("user_id", predicted_items_col, predicted_score_col)
        .explode(predicted_items_col, predicted_score_col)
        .rename({predicted_items_col: _ITEM})
    )

    preds_rel = preds.join(truth, on=["user_id", _ITEM], how="left").fill_null(0)

    gain_expr = (pl.col(relevancy_col) / (pl.col(_RANK) + 1).cast(pl.Float64).log(2)).sum()

    dcg = (
        preds_rel.with_columns(
            pl.col(predicted_score_col)
            .rank("ordinal", descending=True)
            .over("user_id")
            .alias(_RANK)
        )
        .filter(pl.col(_RANK) <= top_k)
        .group_by("user_id")
        .agg(gain_expr.alias("dcg"))
    )

    idcg = (
        preds_rel.with_columns(
            pl.col(relevancy_col).rank("ordinal", descending=True).over("user_id").alias(_RANK)
        )
        .filter(pl.col(_RANK) <= top_k)
        .group_by("user_id")
        .agg(gain_expr.alias("idcg"))
    )

    result = (
        dcg.join(idcg, on="user_id")
        .with_columns(
            pl.when(pl.col("idcg") == 0)
            .then(0.0)
            .otherwise(pl.col("dcg") / pl.col("idcg"))
            .alias("ndcg")
        )
    )
    
    return result["ndcg"].mean()

In [14]:
def precision_recall_at_k(
    df: pl.DataFrame,
    predicted_items_col: str,
    true_items_col: str,
    top_k: int = 15,
) -> tuple[float, float]:
    """
    Вычисляет средние Precision@k и Recall@k по всем пользователям.
    """
    top_k_preds = pl.col(predicted_items_col).list.head(top_k)
    hits_expr = top_k_preds.list.set_intersection(pl.col(true_items_col)).list.len()

    metrics = df.select(
        (hits_expr / top_k).mean().alias("precision"),
        (hits_expr / pl.col(true_items_col).list.len()).mean().alias("recall"),
    )
    return metrics["precision"].item(), metrics["recall"].item()

In [37]:
def build_sequences_streaming(events_df, max_seq_len=30, min_seq_len=3):
    """
    Построение последовательностей с использованием потоковой обработки.
    ВАЖНО: таргет - следующий item после последовательности, а не последний в ней.
    """
    # Сортировка
    events_df = events_df.sort(["user_id", "timestamp"])
    
    sequences = []
    current_user = None
    current_items = []
    current_targets = []
    
    for row in tqdm(events_df.iter_rows(named=True), total=len(events_df), desc="Building sequences"):
        user_id = row["user_id"]
        
        if user_id != current_user:
            # Сохраняем последовательности для предыдущего пользователя
            if len(current_items) >= min_seq_len:
                # Для каждого префикса длины i, таргет - следующий элемент (на позиции i)
                for i in range(min_seq_len, len(current_items)):
                    sequences.append({
                        "user_id": current_user,
                        "item_sequence": current_items[:i],
                        "target": current_targets[i],  # target - следующий item после последовательности
                        "seq_len": i
                    })
            
            # Начинаем нового пользователя
            current_user = user_id
            current_items = [row["item_id"]]
            current_targets = [row["target"]]
        else:
            current_items.append(row["item_id"])
            current_targets.append(row["target"])
            
            # Ограничиваем длину
            if len(current_items) > max_seq_len + 1:  # +1 для таргета
                current_items = current_items[-(max_seq_len + 1):-1]
                current_targets = current_targets[-(max_seq_len + 1):]
    
    # Последний пользователь
    if len(current_items) >= min_seq_len:
        for i in range(min_seq_len, len(current_items)):
            sequences.append({
                "user_id": current_user,
                "item_sequence": current_items[:i],
                "target": current_targets[i],
                "seq_len": i
            })
    
    print(f"Построено {len(sequences)} последовательностей")
    return sequences

In [38]:
timestamp_threshold = events_filtered["timestamp"].quantile(0.8)
train_events = events_filtered.filter(pl.col("timestamp") < timestamp_threshold)
test_events = events_filtered.filter(pl.col("timestamp") >= timestamp_threshold)

In [39]:
train_sequences = build_sequences_streaming(train_events, MAX_SEQ_LEN, MIN_SEQ_LEN)
test_sequences = build_sequences_streaming(test_events, MAX_SEQ_LEN, MIN_SEQ_LEN)


Building sequences: 100%|██████████| 240626/240626 [00:00<00:00, 363799.44it/s]


Построено 74408 последовательностей



Building sequences: 100%|██████████| 60157/60157 [00:00<00:00, 339301.87it/s]

Построено 27877 последовательностей


In [40]:
item_to_idx = {item: idx + 1 for idx, item in enumerate(top_items)} 
idx_to_item = {idx + 1: item for idx, item in enumerate(top_items)}  
num_items = len(top_items) + 1 

In [61]:
class RankingDataset(Dataset):
    """Датасет для обучения ранжированию"""
    
    def __init__(self, sequences, item_to_idx, max_seq_len=30):
        self.max_seq_len = max_seq_len
        self.item_to_idx = item_to_idx
        self.data = []
        
        for seq in sequences:
            item_indices = [item_to_idx.get(item, 0) for item in seq["item_sequence"]]
            seq_len = len(item_indices)
            
            padded_seq = item_indices + [0] * (max_seq_len - seq_len)
            attention_mask = [1] * seq_len + [0] * (max_seq_len - seq_len)
            
            # ГАРАНТИРОВАННОЕ преобразование target в float
            target_raw = seq["target"]
            if isinstance(target_raw, str):
                # Пробуем преобразовать строку в число
                try:
                    target_float = float(target_raw)
                except:
                    target_float = 0.0
            elif isinstance(target_raw, (int, float)):
                target_float = float(target_raw)
            else:
                target_float = 0.0
            
            # Сохраняем уже как float
            self.data.append({
                "input_ids": padded_seq,
                "attention_mask": attention_mask,
                "target": target_float,  # Уже float!
                "user_id": seq["user_id"]
            })
        
        # Дополнительная проверка
        print(f"Dataset создан. Пример target: {self.data[0]['target']} (тип: {type(self.data[0]['target'])})")
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        item = self.data[idx]
        return {
            "input_ids": torch.tensor(item["input_ids"], dtype=torch.long),
            "attention_mask": torch.tensor(item["attention_mask"], dtype=torch.long),
            "target": torch.tensor(item["target"], dtype=torch.float),  # Теперь item["target"] точно float
            "user_id": torch.tensor(item["user_id"], dtype=torch.long)
        }

In [62]:
BATCH_SIZE = 64

train_dataset = RankingDataset(train_sequences, item_to_idx, MAX_SEQ_LEN)
test_dataset = RankingDataset(test_sequences, item_to_idx, MAX_SEQ_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"Train loader: {len(train_loader)} батчей")
print(f"Test loader: {len(test_loader)} батчей")
print(f"Num items (с padding): {num_items}")

Dataset создан. Пример target: 1.0307505993330341 (тип: <class 'float'>)
Dataset создан. Пример target: 1.0307505993330341 (тип: <class 'float'>)
Train loader: 1163 батчей
Test loader: 436 батчей
Num items (с padding): 5001


In [63]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))
    
    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]

In [57]:
class RankingDataset(Dataset):
    """Датасет для обучения ранжированию"""
    
    def __init__(self, sequences, item_to_idx, max_seq_len=30):
        self.max_seq_len = max_seq_len
        self.item_to_idx = item_to_idx
        self.data = []
        
        for seq in sequences:
            item_indices = [item_to_idx.get(item, 0) for item in seq["item_sequence"]]
            seq_len = len(item_indices)
            
            padded_seq = item_indices + [0] * (max_seq_len - seq_len)
            attention_mask = [1] * seq_len + [0] * (max_seq_len - seq_len)
            
            # Преобразуем target в float
            target_value = seq["target"]
            if isinstance(target_value, str):
                try:
                    target_value = float(target_value)
                except (ValueError, TypeError):
                    target_value = 0.0
            elif not isinstance(target_value, (int, float)):
                target_value = 0.0
            
            self.data.append({
                "input_ids": padded_seq,
                "attention_mask": attention_mask,
                "target": target_value,  # Сохраняем как число
                "user_id": seq["user_id"]
            })
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        item = self.data[idx]
        return {
            "input_ids": torch.tensor(item["input_ids"], dtype=torch.long),
            "attention_mask": torch.tensor(item["attention_mask"], dtype=torch.long),
            "target": torch.tensor(float(item["target"]), dtype=torch.float),  # Еще раз преобразуем на всякий случай
            "user_id": torch.tensor(item["user_id"], dtype=torch.long)
        }

In [64]:
class RankingTransformer(nn.Module):
    """Модель для ранжирования: возвращает эмбеддинг пользователя"""
    
    def __init__(self, num_items, d_model=128, nhead=8, num_layers=3, 
                 dim_feedforward=512, dropout=0.3, max_seq_len=50):
        super().__init__()
        
        self.d_model = d_model
        self.num_items = num_items
        
        self.item_embedding = nn.Embedding(num_items, d_model, padding_idx=0)
        self.pos_encoder = PositionalEncoding(d_model, max_seq_len)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward,
            dropout=dropout, activation='gelu', batch_first=True, norm_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.dropout = nn.Dropout(dropout)
        
        self._init_weights()
    
    def _init_weights(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)
    
    def forward(self, input_ids, attention_mask):
        batch_size, seq_len = input_ids.shape
        
        x = self.item_embedding(input_ids) * math.sqrt(self.d_model)
        x = self.pos_encoder(x)
        x = self.dropout(x)
        
        causal_mask = torch.triu(torch.ones(seq_len, seq_len) * float('-inf'), diagonal=1).to(x.device)
        src_key_padding_mask = (attention_mask == 0)
        transformer_out = self.transformer(x, mask=causal_mask, src_key_padding_mask=src_key_padding_mask)
        
        seq_lengths = attention_mask.sum(dim=1) - 1
        batch_indices = torch.arange(batch_size, device=input_ids.device)
        user_embedding = transformer_out[batch_indices, seq_lengths]
        
        return user_embedding
    
    def predict_scores(self, user_embedding, candidate_ids):
        candidate_embeddings = self.item_embedding(candidate_ids)
        scores = torch.bmm(candidate_embeddings, user_embedding.unsqueeze(-1)).squeeze(-1)
        return scores


model = RankingTransformer(
    num_items=num_items,
    d_model=128,
    nhead=8,
    num_layers=3,
    dim_feedforward=512,
    dropout=0.3,
    max_seq_len=MAX_SEQ_LEN
).to(device)

criterion = nn.MSELoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

Model parameters: 1,234,944


In [65]:
model = RankingTransformer(
    num_items=num_items,
    d_model=128,  
    nhead=8,
    num_layers=3,
    dim_feedforward=512,
    dropout=0.3,
    max_seq_len=MAX_SEQ_LEN
).to(device)

criterion = nn.MSELoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)


In [ ]:
EPOCHS = 20
best_val_loss = float('inf')
train_losses = []
val_losses = []

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0
    
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]")
    for batch in progress_bar:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        targets = batch["target"].to(device)
        
        optimizer.zero_grad()
        
        # Получаем эмбеддинг пользователя
        user_embedding = model(input_ids, attention_mask)
        
        # Находим последний item в последовательности (это целевой item)
        seq_lengths = attention_mask.sum(dim=1) - 1
        batch_indices = torch.arange(input_ids.size(0), device=device)
        target_item_ids = input_ids[batch_indices, seq_lengths].unsqueeze(1)
        
        # Предсказываем скор для целевого item
        outputs = model.predict_scores(user_embedding, target_item_ids).squeeze(1)
        
        loss = criterion(outputs, targets)
        loss.backward()
        
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        train_loss += loss.item()
        progress_bar.set_postfix({"loss": f"{loss.item():.4f}"})
    
    avg_train_loss = train_loss / len(train_loader)
    train_losses.append(avg_train_loss)
    
    # Валидация
    model.eval()
    val_loss = 0
    
    with torch.no_grad():
        for batch in tqdm(test_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Val]"):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            targets = batch["target"].to(device)
            
            user_embedding = model(input_ids, attention_mask)
            
            seq_lengths = attention_mask.sum(dim=1) - 1
            batch_indices = torch.arange(input_ids.size(0), device=device)
            target_item_ids = input_ids[batch_indices, seq_lengths].unsqueeze(1)
            
            outputs = model.predict_scores(user_embedding, target_item_ids).squeeze(1)
            val_loss += criterion(outputs, targets).item()
    
    avg_val_loss = val_loss / len(test_loader)
    val_losses.append(avg_val_loss)
    
    scheduler.step(avg_val_loss)
    current_lr = optimizer.param_groups[0]['lr']
    
    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    print(f"  Train Loss: {avg_train_loss:.6f}")
    print(f"  Val Loss:   {avg_val_loss:.6f}")
    print(f"  LR: {current_lr:.6f}")
    
    # Сохраняем лучшую модель
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), "best_ranking_model.pt")
        print("  -> Сохранена лучшая модель!")


Epoch 1/20 [Train]: 100%|██████████| 1163/1163 [00:18<00:00, 62.28it/s, loss=0.7235]

Epoch 1/20 [Val]: 100%|██████████| 436/436 [00:01<00:00, 255.22it/s]



Epoch 1/20
  Train Loss: 0.619579
  Val Loss:   0.768957
  LR: 0.001000
  -> Сохранена лучшая модель!



Epoch 2/20 [Train]:  59%|█████▉    | 690/1163 [00:10<00:06, 69.82it/s, loss=0.6072]

Epoch 8/20 [Train]: 100%|██████████| 1163/1163 [00:18<00:00, 61.82it/s, loss=0.0994]

Epoch 8/20 [Val]: 100%|██████████| 436/436 [00:01<00:00, 264.81it/s]



Epoch 8/20
  Train Loss: 0.418946
  Val Loss:   0.911109
  LR: 0.000500



Epoch 9/20 [Train]: 100%|██████████| 1163/1163 [00:17<00:00, 66.87it/s, loss=0.3949]

Epoch 9/20 [Val]: 100%|██████████| 436/436 [00:01<00:00, 266.14it/s]


...


Epoch 11/20 [Train]: 100%|██████████| 1163/1163 [00:17<00:00, 67.22it/s, loss=0.4136]

Epoch 11/20 [Val]: 100%|██████████| 436/436 [00:01<00:00, 268.69it/s]



Epoch 11/20
  Train Loss: 0.346786
  Val Loss:   0.896463
  LR: 0.000250



Epoch 12/20 [Train]: 100%|██████████| 1163/1163 [00:17<00:00, 67.40it/s, loss=0.0518]

Epoch 12/20 [Val]: 100%|██████████| 436/436 [00:01<00:00, 264.38it/s]



Epoch 12/20
  Train Loss: 0.333697
  Val Loss:   0.896259
  LR: 0.000250



Epoch 13/20 [Train]:  23%|██▎       | 266/1163 [00:04<00:13, 66.37it/s, loss=0.0896]

In [70]:
# Загружаем лучшую модель
model.load_state_dict(torch.load("best_ranking_model.pt"))
model.eval()

# Получаем истинные взаимодействия для тестового периода
test_true_interactions = (
    test_events
    .group_by("user_id")
    .agg([
        pl.col("item_id").alias("true_items"),
        pl.col("target").cast(pl.Float64).alias("relevancy")
    ])
)

# Функция для получения эмбеддингов пользователей
def get_user_embeddings(model, dataloader):
    model.eval()
    user_embeddings = []
    user_ids = []
    
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Computing user embeddings"):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            embeddings = model(input_ids, attention_mask).cpu().numpy()
            
            user_embeddings.append(embeddings)
            user_ids.extend(batch["user_id"].numpy())
    
    return np.vstack(user_embeddings), user_ids

print("Computing user embeddings...")
user_embeddings, user_ids = get_user_embeddings(model, test_loader)
print(f"Computed embeddings for {len(user_ids)} users")

# Получаем эмбеддинги всех items
all_item_ids = torch.arange(num_items, device=device).unsqueeze(0)
with torch.no_grad():
    all_item_embeddings = model.item_embedding(all_item_ids).squeeze(0).cpu().numpy()

# Для каждого пользователя ранжируем все items
TOP_K = 15
all_predictions = []

for i, (user_id, user_emb) in enumerate(tqdm(zip(user_ids, user_embeddings), total=len(user_ids), desc="Ranking items")):
    scores = np.dot(all_item_embeddings, user_emb)
    scores[0] = -np.inf
    
    top_k_indices = np.argsort(scores)[-TOP_K:][::-1]
    
    predicted_items = [idx_to_item.get(idx, idx) for idx in top_k_indices if idx != 0]
    predicted_scores = [float(scores[idx]) for idx in top_k_indices if idx != 0]
    
    all_predictions.append({
        "user_id": int(user_id),
        "predicted_items": predicted_items[:TOP_K],
        "predicted_scores": predicted_scores[:TOP_K]
    })

predictions_df = pl.DataFrame(all_predictions)
print(f"Predictions for {len(predictions_df)} users")

# Объединяем с истинными взаимодействиями
user_metrics_df = predictions_df.join(test_true_interactions, on="user_id", how="inner")
print(f"Users with both predictions and true interactions: {len(user_metrics_df)}")

# Расчет метрик
ndcg_score = ndcg_at_k_vectorized(
    user_metrics_df, 
    relevancy_col="relevancy",
    true_items_col="true_items",
    predicted_items_col="predicted_items",
    predicted_score_col="predicted_scores",
    top_k=TOP_K
)

precision_score, recall_score = precision_recall_at_k(
    user_metrics_df, 
    predicted_items_col="predicted_items", 
    true_items_col="true_items", 
    top_k=TOP_K
)

# Исправленный Hit Rate
hit_rate_df = user_metrics_df.with_columns(
    (pl.col("predicted_items").list.head(TOP_K)
     .list.set_intersection(pl.col("true_items"))
     .list.len() > 0).alias("hit")
)
hit_rate = hit_rate_df["hit"].mean()

results = pl.DataFrame({
    "top_k": [TOP_K],
    "ndcg": [ndcg_score],
    "precision": [precision_score],
    "recall": [recall_score],
    "hit_rate": [hit_rate],
    "num_users": [len(user_metrics_df)],
    "num_items": [num_items - 1]
})

print(results)
print("="*80)

print(f"\nДЕТАЛЬНЫЙ АНАЛИЗ:")
print(f"  NDCG@{TOP_K}:      {ndcg_score:.6f}")
print(f"  Precision@{TOP_K}: {precision_score:.6f}")
print(f"  Recall@{TOP_K}:    {recall_score:.6f}")
print(f"  Hit Rate@{TOP_K}:  {hit_rate:.6f}")
print(f"  Users: {len(user_metrics_df)}")
print(f"  Items: {num_items - 1}")
print("="*80)

Computing user embeddings...



Computing user embeddings: 100%|██████████| 436/436 [00:01<00:00, 245.92it/s]


Computed embeddings for 27877 users



Ranking items: 100%|██████████| 27877/27877 [00:05<00:00, 5034.34it/s]


Predictions for 27877 users
Users with both predictions and true interactions: 27877
shape: (1, 7)
┌───────┬──────────┬───────────┬──────────┬──────────┬───────────┬───────────┐
│ top_k ┆ ndcg     ┆ precision ┆ recall   ┆ hit_rate ┆ num_users ┆ num_items │
│ ---   ┆ ---      ┆ ---       ┆ ---      ┆ ---      ┆ ---       ┆ ---       │
│ i64   ┆ f64      ┆ f64       ┆ f64      ┆ f64      ┆ i64       ┆ i64       │
╞═══════╪══════════╪═══════════╪══════════╪══════════╪═══════════╪═══════════╡
│ 15    ┆ 0.353719 ┆ 0.03889   ┆ 0.023839 ┆ 0.532087 ┆ 27877     ┆ 5000      │
└───────┴──────────┴───────────┴──────────┴──────────┴───────────┴───────────┘

ДЕТАЛЬНЫЙ АНАЛИЗ:
  NDCG@15:      0.353719
  Precision@15: 0.038890
  Recall@15:    0.023839
  Hit Rate@15:  0.532087
  Users: 27877
  Items: 5000
